# Ticket Quality

This notebook reads cached ticket quality flags from DuckDB and highlights tickets where the interaction pattern suggests weak response discipline.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

import plotly.express as px

db_path = Path('/Users/micahcooper/dynamix-manager/data/analytics.duckdb')
connection = duckdb.connect(str(db_path), read_only=True)


In [2]:
ticket_quality_flags = connection.execute(
    "select * from ticket_quality_flags"
).fetchdf()
if 'last_public_interaction_at' in ticket_quality_flags.columns:
    ticket_quality_flags['last_public_interaction_at'] = pd.to_datetime(ticket_quality_flags['last_public_interaction_at'], utc=True, errors='coerce')
if 'last_private_interaction_at' in ticket_quality_flags.columns:
    ticket_quality_flags['last_private_interaction_at'] = pd.to_datetime(ticket_quality_flags['last_private_interaction_at'], utc=True, errors='coerce')
for column, default in {
    'client_last_interaction_flag': False,
    'it_follow_up_without_client_response_flag': False,
    'stale_public_update_flag': False,
    'private_activity_since_last_public_flag': False,
    'stale_public_update_business_days': 0,
    'last_private_interaction_at': pd.NaT,
    'last_private_interaction_by': pd.NA,
}.items():
    if column not in ticket_quality_flags.columns:
        ticket_quality_flags[column] = default
ticket_quality_flags.head()

,ticket_id,ticket_title,status_name,service_name,team_name,assignee_name,requestor_name,requestor_uid,created_at,modified_at,...,last_private_interaction_by,last_public_interaction_actor_type,last_public_interaction_by,client_last_interaction_flag,it_follow_up_streak,it_follow_up_without_client_response_flag,interaction_count,stale_public_update_business_days,stale_public_update_flag,private_activity_since_last_public_flag


## Executive Summary

In [3]:
pd.DataFrame([
    {'metric': 'Tickets reviewed', 'value': len(ticket_quality_flags)},
    {'metric': 'Client last interaction', 'value': int(ticket_quality_flags['client_last_interaction_flag'].fillna(False).sum())},
    {'metric': 'Repeated IT follow-up', 'value': int(ticket_quality_flags['it_follow_up_without_client_response_flag'].fillna(False).sum())},
    {'metric': 'Stale public update', 'value': int(ticket_quality_flags['stale_public_update_flag'].fillna(False).sum())},
    {'metric': 'Private activity since last public update', 'value': int(ticket_quality_flags['private_activity_since_last_public_flag'].fillna(False).sum())},
])

,metric,value
0,Tickets reviewed,0
1,Client last interaction,0
2,Repeated IT follow-up,0
3,Stale public update,0
4,Private activity since last public update,0


In [4]:
quality_summary_chart = pd.DataFrame([
    {'metric': 'Client last', 'tickets': int(ticket_quality_flags['client_last_interaction_flag'].fillna(False).sum())},
    {'metric': 'Repeated IT follow-up', 'tickets': int(ticket_quality_flags['it_follow_up_without_client_response_flag'].fillna(False).sum())},
    {'metric': 'Stale public update', 'tickets': int(ticket_quality_flags['stale_public_update_flag'].fillna(False).sum())},
    {'metric': 'Private after public', 'tickets': int(ticket_quality_flags['private_activity_since_last_public_flag'].fillna(False).sum())},
])
px.bar(
    quality_summary_chart,
    x='metric',
    y='tickets',
    title='Open Ticket Quality Flags',
)


## Client Last Interaction

In [5]:
ticket_quality_flags.loc[
    ticket_quality_flags['client_last_interaction_flag'].fillna(False),
    ['ticket_id', 'ticket_title', 'team_name', 'service_name', 'requestor_name', 'last_public_interaction_at', 'last_public_interaction_by'],
].sort_values('last_public_interaction_at', ascending=False).head(50)

,ticket_id,ticket_title,team_name,service_name,requestor_name,last_public_interaction_at,last_public_interaction_by


## Repeated IT Follow-Up

In [6]:
ticket_quality_flags.loc[
    ticket_quality_flags['it_follow_up_without_client_response_flag'].fillna(False),
    ['ticket_id', 'ticket_title', 'team_name', 'service_name', 'requestor_name', 'it_follow_up_streak', 'last_public_interaction_at', 'last_public_interaction_by'],
].sort_values(['it_follow_up_streak', 'last_public_interaction_at'], ascending=[False, False]).head(50)

,ticket_id,ticket_title,team_name,service_name,requestor_name,it_follow_up_streak,last_public_interaction_at,last_public_interaction_by


## Stale Public Update

In [7]:
ticket_quality_flags.loc[
    ticket_quality_flags['stale_public_update_flag'].fillna(False),
    ['ticket_id', 'ticket_title', 'team_name', 'service_name', 'requestor_name', 'stale_public_update_business_days', 'last_public_interaction_at', 'last_public_interaction_by'],
].sort_values(['stale_public_update_business_days', 'last_public_interaction_at'], ascending=[False, False]).head(50)

,ticket_id,ticket_title,team_name,service_name,requestor_name,stale_public_update_business_days,last_public_interaction_at,last_public_interaction_by


## Private Activity Since Last Public Update

In [8]:
ticket_quality_flags.loc[
    ticket_quality_flags['private_activity_since_last_public_flag'].fillna(False),
    ['ticket_id', 'ticket_title', 'team_name', 'service_name', 'requestor_name', 'last_private_interaction_at', 'last_private_interaction_by', 'last_public_interaction_by'],
].sort_values(['last_private_interaction_at', 'ticket_id'], ascending=[False, True]).head(50)

,ticket_id,ticket_title,team_name,service_name,requestor_name,last_private_interaction_at,last_private_interaction_by,last_public_interaction_by
